# AC-OPF Relaxation Test

Solves a single AC-OPF instance at nominal demand using four methods and compares their optimal values:
- **Local solver** (pandapower / IPOPT) — true AC-OPF, non-convex
- **SOCP relaxation** (Jabr) — convex lower bound via second-order cone constraints
- **Chordal SDP** (Madani-Ashraphijuo-Lavaei) — sparse SDP via tree decomposition; same bound as dense SDP but tractable for large cases
- **SDP relaxation** (Lavaei-Low, real 2n×2n) — tightest convex lower bound, dense formulation

In [1]:
import pathlib, sys, time
PROJECT_ROOT = pathlib.Path(".").resolve().parents[0]
sys.path.insert(0, str(PROJECT_ROOT))

from problems.acopf.network import load_network
from problems.acopf.problem import solve_local, solve_relaxation

In [2]:
net, nd = load_network("case1354pegase")# v_min=0.90, v_max=1.10)

print(f"Buses:       {nd.n_buses}")
print(f"Branches:    {len(nd.branch_from)}")
print(f"Generators:  {nd.n_gens}")
print(f"Loads:       {nd.n_loads}")
print(f"Base MVA:    {nd.baseMVA}")
print(f"Voltage bounds: [{nd.v_min.min():.2f}, {nd.v_max.max():.2f}] pu")

Buses:       1354
Branches:    1710
Generators:  260
Loads:       621
Base MVA:    100.0
Voltage bounds: [0.70, 1.30] pu


## Build the nominal demand vector

Use `α = 1`, `η = 1` everywhere — i.e. exactly the reference demand stored in the network.

In [3]:
import numpy as np

p_nominal = np.concatenate([nd.pd_nominal, nd.qd_nominal])  # [Pd (MW), Qd (MVar)]
print(f"Total nominal Pd: {nd.pd_nominal.sum():.1f} MW")
print(f"Total nominal Qd: {nd.qd_nominal.sum():.1f} MVar")

Total nominal Pd: 74146.0 MW
Total nominal Qd: 13717.2 MVar


## Local solver (Pyomo / IPOPT)

In [4]:
t0 = time.perf_counter()
local_val, local_res = solve_local(p_nominal, args={"nd": nd})
local_time = time.perf_counter() - t0
print(f"Local (AC-OPF):  {local_val:.4f}  [converged: {local_res['success']},  time: {local_time:.2f}s]")

Local (AC-OPF):  75159.2161  [converged: True,  time: 1.62s]


## SOCP relaxation (Jabr)

In [14]:
t0 = time.perf_counter()
socp_val, socp_res = solve_relaxation(p_nominal, args={"nd": nd, "relaxation": "socp"})
socp_time = time.perf_counter() - t0
print(f"SOCP relaxation: {socp_val:.4f}  [status: {socp_res['status']}, exact: {socp_res['exact']},  time: {socp_time:.2f}s]")

SOCP relaxation: 731904.0869  [status: optimal, exact: False,  time: 1.07s]


## Chordal SDP relaxation (sparse / tree decomposition)

Uses a greedy minimum-fill elimination ordering to decompose the network graph into clique bags, then imposes one small complex Hermitian PSD constraint per bag instead of a single dense 2n×2n PSD constraint.  This makes large cases (case118, case300) tractable.

Key quantities reported:
- **Treewidth** — maximum bag size minus 1; bounds the size of the largest PSD block.
- **Bags** — number of eliminated vertices (equals n\_buses); most bags are tiny.
- **Exact** — True if every bag's solution matrix is rank-1 (global optimality certificate).

In [ ]:
import time
from problems.acopf.chordal import greedy_elimination, unique_pairs_in_bags

# Report tree-decomposition stats before solving.
bags, tw = greedy_elimination(nd.branch_from, nd.branch_to, nd.n_buses)
bag_sizes = [len(b) for b in bags]
pairs = unique_pairs_in_bags(bags)
print(f"Treewidth (upper bound): {tw}")
print(f"Bags:                    {len(bags)}")
print(f"Max bag size:            {max(bag_sizes)}  (= treewidth + 1)")
print(f"Unique off-diag pairs:   {len(pairs)}  (scalar VV variables)")
print(f"Bag size histogram:      { {s: bag_sizes.count(s) for s in sorted(set(bag_sizes))} }")

t0 = time.perf_counter()
chordal_val, chordal_res = solve_relaxation(
    p_nominal,
    args={"nd": nd, "relaxation": "chordal_sdp"},
)
chordal_time = time.perf_counter() - t0

print(f"\nChordal SDP: {chordal_val:.4f}  "
      f"[status: {chordal_res['status']}, exact: {chordal_res['exact']}, "
      f"time: {chordal_time:.2f}s]")

Treewidth (upper bound): 12
Bags:                    1354
Max bag size:            13  (= treewidth + 1)
Unique off-diag pairs:   2714  (scalar VV variables)
Bag size histogram:      {1: 1, 2: 660, 3: 434, 4: 116, 5: 43, 6: 23, 7: 29, 8: 26, 9: 11, 10: 6, 11: 3, 12: 1, 13: 1}
  [chordal_sdp] treewidth=12, bags=1354


/opt/anaconda3/envs/nn4opt/lib/python3.11/site-packages/cvxpy/expressions/constants/constant.py:52: UserWarning: Initializing a Constant with a nested list is undefined behavior. Consider using a numpy array instead.
  warnings.warn(NESTED_LIST_WARNING)


## SDP relaxation (Lavaei-Low, real 2n×2n)

> **Note**: the dense SDP lifts to a 2n×2n matrix, so for n=118 this is a 236×236 PSD constraint — expect a long solve time (minutes to hours).  For large cases, use the **Chordal SDP** below instead and skip this cell.

In [ ]:
t0 = time.perf_counter()
sdp_val, sdp_res = solve_relaxation(p_nominal, args={"nd": nd, "relaxation": "sdp"})
sdp_time = time.perf_counter() - t0
print(f"SDP relaxation:  {sdp_val:.4f}  [status: {sdp_res['status']}, exact: {sdp_res['exact']},  time: {sdp_time:.2f}s]")

In [9]:
sdp_val, sdp_res, sdp_time = np.nan, {"exact": None}, np.nan

## Comparison

In [10]:
print(f"{'Method':<24} {'Value':>12} {'Gap vs local':>14} {'Exact':>7} {'Time (s)':>10}")
print("-" * 72)
for label, val, exact, t in [
    ("Local (AC-OPF)",  local_val,    None,                   local_time),
    ("SOCP",            socp_val,     socp_res["exact"],       socp_time),
    ("Chordal SDP",     chordal_val,  chordal_res["exact"],    chordal_time),
    ("SDP (dense)",     sdp_val,      sdp_res.get("exact"),    sdp_time),
]:
    gap       = 100 * (local_val - val) / local_val if local_val else float("nan")
    exact_str = "—" if exact is None else str(exact)
    t_str     = f"{t:.2f}" if t is not None else "—"
    print(f"{label:<24} {val:>12.4f} {gap:>13.4f}%  {exact_str:>7} {t_str:>10}")

print()
if sdp_val and chordal_val:
    print(f"Chordal vs dense SDP delta: {abs(chordal_val - sdp_val):.4f} $/hr")

Method                          Value   Gap vs local   Exact   Time (s)
------------------------------------------------------------------------
Local (AC-OPF)            732908.1069        0.0000%        —       0.31
SOCP                      731904.0869        0.1370%    False       1.05
Chordal SDP               732877.5341        0.0042%    False      20.55
SDP (dense)                       nan           nan%        —        nan

Chordal vs dense SDP delta: nan $/hr
